# Pytorch TabNet: Deep Learning for Tabular Data
TabNet was created by Google AI to combine the specific feature-selection capabilities of Tree-based models (like LightGBM/XGBoost) with the representation power of Deep Learning.

Unlike TabPFN, **TabNet does not need to compress the dataset into 100 features.** It natively takes all 2,381 raw EMBER features and uses a Sparse Sequential Attention mechanism to actively decide which features are useful for classifying malware, and ignores completely useless bytes (like unused API headers or empty sequence padding).

In [5]:
import os
import numpy as np
import torch
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
print("TabNet successfully imported!")

TabNet successfully imported!


In [6]:
# --- 1. LOAD A SUBSTANTIAL DATASET ---
DATASET_DIR = r"Z:\ember2024_train_data"
ndim = 2381

# Dropping slightly from 100,000 to 60,000 to prevent your computer's RAM from crashing 
# while scaling the extreme 2,381 feature depth
nrows_to_read = 60000 

X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

print("Reading chunk from disk...")
with open(X_path, 'rb') as f_x:
    X_chunk = np.frombuffer(f_x.read(nrows_to_read * ndim * 4), dtype=np.float32).reshape(nrows_to_read, ndim)
with open(y_path, 'rb') as f_y:
    y_chunk = np.frombuffer(f_y.read(nrows_to_read * 4), dtype=np.int32)

# Filter out unlabeled (-1)
valid_idx = np.where(y_chunk != -1)[0]
X_chunk = X_chunk[valid_idx]
y_chunk = y_chunk[valid_idx]

print(f"Extracted {len(y_chunk)} Total Labeled Files.")

print("Splitting into Train / Validation (80/20)...")
X_train, X_valid, y_train, y_valid = train_test_split(
    X_chunk, y_chunk, test_size=0.2, random_state=42, stratify=y_chunk
)

del X_chunk
del y_chunk
import gc
gc.collect()

print(f"Train Shape: {X_train.shape} | Valid Shape: {X_valid.shape}")

Reading chunk from disk...
Extracted 60000 Total Labeled Files.
Splitting into Train / Validation (80/20)...
Train Shape: (48000, 2381) | Valid Shape: (12000, 2381)


In [7]:
# --- 2. DATA PREPROCESSING (SCALING) ---
import gc
from sklearn.preprocessing import StandardScaler

# Clean missing values first. Modify in-place to save memory.
np.nan_to_num(X_train, copy=False)
np.nan_to_num(X_valid, copy=False)

print("Applying Robust Scaling (to ignore outliers) followed by Standard Scaling...")
robust = RobustScaler()
X_train_scaled = robust.fit_transform(X_train)
X_valid_scaled = robust.transform(X_valid)

# CRITICAL FIX: RobustScaler limits the IQR but allows extreme outliers to still exist 
# (sometimes at values of 4,000+). TabNet's Batch Normalization collapses if inputs aren't mapped near -1 to 1.
# We must squish the robust-scaled data into standard Normal Distribution.
standard = StandardScaler()
X_train_scaled = standard.fit_transform(X_train_scaled).astype(np.float32)
X_valid_scaled = standard.transform(X_valid_scaled).astype(np.float32)

# Delete unscaled arrays to free up RAM before starting the Deep Learning phase
del X_train
del X_valid
gc.collect()

print(f"Data ready for TabNet. Feature count maintained at: {X_train_scaled.shape[1]}")

Applying Robust Scaling (to ignore outliers) followed by Standard Scaling...
Data ready for TabNet. Feature count maintained at: 2381


In [ ]:
# --- 3. DEFINING AND TRAINING THE DEEP LEARNING MODEL ---
from pytorch_tabnet.callbacks import Callback

# A custom mechanism to checkpoint/save the model to disk every X epochs
class CheckpointCallback(Callback):
    def __init__(self, checkpoint_dir="tabnet_checkpoints", save_every=2):
        self.checkpoint_dir = checkpoint_dir
        self.save_every = save_every
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_every == 0:
            save_path = os.path.join(self.checkpoint_dir, f"model_epoch_{epoch+1}")
            self.trainer.save_model(save_path)
            print(f"  --> [CHECKPOINT] Safely wrote progress to {save_path}.zip")

device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"PyTorch Execution Device Details: {device}")

print("Initializing Google's TabNet Architecture...")
# The network is STILL getting hammered into a 50% guess 
# because the Learning Rate of 0.02 is simply too violent for the standardized data.
clf = TabNetClassifier(
    n_d=16,                  
    n_a=16,                  
    n_steps=3,               
    gamma=1.3, 
    n_independent=2, 
    n_shared=2,
    lambda_sparse=1e-4,       # Add a tiny bit of sparsity back to help it ignore zeros
    momentum=0.02, 
    clip_value=2.,           
    optimizer_fn=torch.optim.Adam,
    # CRITICAL FIX: The Learning Rate must be microscopic (2e-4) to safely descend 
    # the complex 2,381 feature gradient without exploding.
    optimizer_params=dict(lr=2e-4),  
    scheduler_params={"step_size": 10, "gamma": 0.90},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',      # Reverted to entmax since we lowered the LR drastically
    device_name='auto' 
)

print("Starting Model Backpropagation...")
clf.fit(
    X_train=X_train_scaled, y_train=y_train,
    eval_set=[(X_train_scaled, y_train), (X_valid_scaled, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['auc', 'accuracy'],
    max_epochs=100,            
    patience=15,               
    batch_size=1024,         
    virtual_batch_size=128,  
    num_workers=0,
    drop_last=False,
    callbacks=[CheckpointCallback(save_every=1)] 
)

PyTorch Execution Device Details: CPU
Initializing Google's TabNet Architecture...
Starting Model Backpropagation...


epoch 0  | loss: 0.71617 | train_auc: 0.49958 | train_accuracy: 0.50623 | valid_auc: 0.4976  | valid_accuracy: 0.50533 |  0:02:17s
Successfully saved model at tabnet_checkpoints\model_epoch_1.zip
  --> [CHECKPOINT] Safely wrote progress to tabnet_checkpoints\model_epoch_1.zip
epoch 1  | loss: 0.6944  | train_auc: 0.50115 | train_accuracy: 0.49929 | valid_auc: 0.50895 | valid_accuracy: 0.5075  |  0:04:30s
Successfully saved model at tabnet_checkpoints\model_epoch_2.zip
  --> [CHECKPOINT] Safely wrote progress to tabnet_checkpoints\model_epoch_2.zip
epoch 2  | loss: 0.69424 | train_auc: 0.49904 | train_accuracy: 0.49525 | valid_auc: 0.50196 | valid_accuracy: 0.49217 |  0:06:34s
Successfully saved model at tabnet_checkpoints\model_epoch_3.zip
  --> [CHECKPOINT] Safely wrote progress to tabnet_checkpoints\model_epoch_3.zip
epoch 3  | loss: 0.69379 | train_auc: 0.50064 | train_accuracy: 0.49383 | valid_auc: 0.50172 | valid_accuracy: 0.4925  |  0:08:43s
Successfully saved model at tabnet_che

KeyboardInterrupt: 

In [ ]:
# --- 4. EVALUATE FINAL TEST ACCURACY ---
print("Evaluating Accuracy and ROC_AUC on 20,000 Unseen Samples...")
y_pred = clf.predict(X_valid_scaled)
y_prob = clf.predict_proba(X_valid_scaled)[:, 1]

acc = accuracy_score(y_valid, y_pred)
auc = roc_auc_score(y_valid, y_prob)

print(f"\n===== [ GOOGLE TABNET ] =====")
print(f"Final Accuracy: {acc*100:.2f}%")
print(f"Final AUC:      {auc:.4f}")

Evaluating Accuracy and ROC_AUC on 20,000 Unseen Samples...

===== [ GOOGLE TABNET ] =====
Final Accuracy: 50.98%
Final AUC:      0.5028
